# freeCAM persistent runtime: one live model, many continuations

This Notebook keeps one MPI model alive across cells. It launches MPI once and reuses rank-local in-memory snapshots for sequential branches. Jupyter displays each cell's execution duration, so the code contains no manual timers.

## 1. Execution paths

```text
Single-model persistent runtime              Legacy multi-job
one launcher + one model worker              base MPI job
one mpiexec → one live MPI model             ├── child MPI job: control
base → rank-local retained bytes              ├── child MPI job: no-kessler
                                             └── child MPI job: warm
  ├── restore control sequentially
  ├── restore warm sequentially
  └── restore no-kessler sequentially

restore = local memory decode, no disk         fork = checkpoint + new MPI startup
```

Only one branch is live at a time, so the main path needs 24 MPI ranks instead of 96 for a four-model experiment. The optional legacy cell performs the corresponding multi-job workload, and its cell duration includes queue, MPI startup, checkpoint, and model costs.

## 2. Configure Dask

The normal API plans one live MPI model and one retained snapshot automatically. There is no user-visible `plan_pool()` step. For one uninterrupted run, the shortest form is `with experiments.model('base') as base:`. This Notebook uses `experiments.runtime()` only because later cells close and sequentially restore several models from the same retained state.

In [1]:
from datetime import datetime
from pathlib import Path
import os
import shutil

import numpy as np
import freecam
from dask.distributed import Client
from freecam import DaskExperimentClient

repo = Path('/glade/work/ruitong/freeCAM')
scratch = Path(os.environ.get('SCRATCH', '/glade/derecho/scratch/ruitong'))
config_path = repo / 'configs/fkessler_model.yaml'
reference_atm_in = repo / 'reference/cases/FKESSLER_ne3pg3_gnu_24x50/CaseDocs/atm_in'
stamp = datetime.now().strftime('%Y%m%d-%H%M%S')
experiment_root = scratch / 'freecam/persistent_pool_trials' / stamp
initial_run_dir = experiment_root / 'initial-run'
initial_run_dir.mkdir(parents=True, exist_ok=False)
shutil.copy2(reference_atm_in, initial_run_dir / 'atm_in')

# Use allocation mode when this Jupyter server is running inside qsub -I.
execution_mode = 'allocation' if os.environ.get('PBS_JOBID') else 'pbs'
client = Client(
    processes=False,
    # One launcher worker plus one live-model controller.
    n_workers=2,
    threads_per_worker=1,
    dashboard_address=None,
)
experiments = DaskExperimentClient(
    client,
    config=config_path,
    initial_run_dir=initial_run_dir,
    run_root=experiment_root / 'models',
    python_executable=repo / '.venv/bin/python',
    execution_mode=execution_mode,
)
{
    'freecam': freecam.__version__,
    'execution_mode': execution_mode,
    'run_root': str(experiment_root),
    'default_runtime': 'one live model; one retained snapshot',
}

{'pycam_sima': '0.19.0',
 'execution_mode': 'pbs',
 'run_root': '/glade/derecho/scratch/ruitong/pycam-sima/persistent_pool_trials/20260801-021434',
 'resource_plan': {'available_nodes': 1,
  'available_cpus': 24,
  'available_memory_bytes': 85899345920,
  'ranks_per_model': 24,
  'model_slots': 1,
  'world_size': 24,
  'slot_placements': ((0,
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    10,
    11,
    12,
    13,
    14,
    15,
    16,
    17,
    18,
    19,
    20,
    21,
    22,
    23),),
  'estimated_model_bytes': 39555323,
  'memory_per_model_bytes': 39555323,
  'reserve_bytes': 12884901888,
  'cpus_per_node': 24,
  'memory_per_node_bytes': 85899345920,
  'threads_per_rank': 1,
  'static_state_bytes': 35959384,
  'dynamic_field_budget_bytes': 3595939,
  'retained_snapshots': 1,
  'retained_snapshot_budget_bytes': 39555323,
  'placement': 'auto',
  'resource_source': 'override',
  'available_memory': '80GiB',
  'estimated_model_memory': '39555323B',
  'mem

PyCAM-SIMA pool submitted as 6971993.desched1; waiting for 1 x 24 MPI ranks ...


## 3. Start the runtime and base once

The runtime and base are stored as normal Python variables, so later cells reuse the same live MPI processes and StatePool. `experiments.runtime()` performs the hidden single-model resource plan with one retained snapshot by default. Do not rerun this cell without first running the cleanup cell.

In [2]:
runtime = experiments.runtime('cam-runtime')

base = runtime.model('base')

runtime_started = runtime.status
base_started = base.status
assert runtime_started['mpi_launch_count'] == 1
{
    'mpi_launch_count': runtime_started['mpi_launch_count'],
    'runtime_mpi_launch_id': runtime_started.get('pool_mpi_launch_id'),
    'pbs_job_id': runtime_started.get('pbs_job_id'),
    'launcher_worker': runtime.worker,
    'base_worker': base.worker,
    'base_slot': base.slot_id,
    'scheduler': runtime.scheduler_status,
    'base_status': base_started,
    'runtime_state': runtime.slots,
}

{'mpi_launch_count': 1,
 'pool_mpi_launch_id': 'cam-pool-derecho7-56359-1785572074728394301',
 'pbs_job_id': None,
 'launcher_worker': 'inproc://128.117.211.176/56359/4',
 'base_worker': 'inproc://128.117.211.176/56359/6',
 'base_slot': 0,
 'scheduler': {'launcher_worker': 'inproc://128.117.211.176/56359/4',
  'model_workers': {'base': 'inproc://128.117.211.176/56359/6'},
  'available_workers': ('inproc://128.117.211.176/56359/4',
   'inproc://128.117.211.176/56359/6'),
  'actor_layout': 'model-per-worker',
  'worker_policy': 'exclusive'},
 'base_status': ModelStatus(name='base', running=True, ranks=24, step=0, native_calls=0, mpi_launch_count=1, worker_host='derecho7', worker_pid=56359, launch_mode='pbs', pbs_job_id='6971993.desched1', outer_pbs_job_id=None, field_count=360, snapshot_transport='initialization', run_dir=PosixPath('/glade/derecho/scratch/ruitong/pycam-sima/persistent_pool_trials/20260801-021434/models/cam-pool/base/run'), history_dir=PosixPath('/glade/derecho/scratch/ru

## 4. A later cell reuses the same base memory

No new `mpiexec`, no model restart, and no checkpoint restore occurs here. This cell demonstrates dynamic field creation/removal and runtime Fortran plugin loading before retention.

In [3]:
launches_before_reuse = runtime.status['mpi_launch_count']
base.advance(steps=2)

base.fields.create(
    'experiment_tracer',
    dims=('column', 'level'),
    units='kg kg-1',
    initial=0.0,
)
base.fields.create('temporary_probe', dims=('column',), initial=1.0)
deleted_probe = base.fields.delete('temporary_probe')
installed_plugin = base.physics.install(
    source=repo / 'examples/plugins/runtime_temperature_offset/device.yaml',
    project_root=repo,
    after='kessler',
    inputs={
        'runtime_plugin_temperature': 240.0,
        'runtime_plugin_temperature_increment': 1.5,
    },
)
plugin_field = base.fields.ccpp_runtime_plugin_temperature
plugin_before = plugin_field.stats(rank=0)
base.physics.scheme('runtime_temperature_offset', group='before').run()
plugin_after = plugin_field.stats(rank=0)
assert np.isclose(plugin_after['mean'] - plugin_before['mean'], 1.5)
assert runtime.status['mpi_launch_count'] == launches_before_reuse == 1
{
    'base_step': base.status.step,
    'same_mpi_launch_count': runtime.status['mpi_launch_count'],
    'deleted_dynamic_field': deleted_probe['standard_name'],
    'installed_plugin': installed_plugin['name'],
    'plugin_increment': plugin_after['mean'] - plugin_before['mean'],
}

{'base_step': 2,
 'same_mpi_launch_count': 1,
 'deleted_dynamic_field': 'temporary_probe',
 'installed_plugin': 'runtime_temperature_offset',
 'plugin_increment': 1.5}

## 5. Insert a Notebook Python function into the live suite

`install_python()` serializes this trusted function and installs it on every MPI rank of the persistent base model. `reads` are read-only views, `writes` are writable rank-local views, and only declared fields are visible. Custom values are keyword-only arguments declared with `parameters={...}`. `run(increment=...)` overrides one call, while `process.parameters[...] = ...` changes later calls and is retained by snapshots. The cell below also demonstrates `disable()`, `enable()`, `move()`, and `remove()`; after removal it reinstalls the callback so retained-memory and checkpoint restore can continue using it.

In [4]:
def notebook_tracer_source(fields, context, *, increment):
    tracer = fields['experiment_tracer']
    tracer[...] += increment

python_process = base.physics.install_python(
    notebook_tracer_source,
    name='notebook_tracer_source',
    group='physics_before_coupler',
    after='kessler',
    writes=('experiment_tracer',),
    parameters={'increment': 1800.0e-6},
)

# Run only this process: the model clock does not advance.
tracer_before = base.fields.experiment_tracer.get(rank=0)
step_before = base.status.step
python_process.run()
tracer_after = base.fields.experiment_tracer.get(rank=0)
assert base.status.step == step_before
assert np.allclose(
    tracer_after,
    tracer_before + 1800.0e-6,
    rtol=0.0,
    atol=np.spacing(1800.0e-6),
)

# Override only this call; the persistent default remains unchanged.
override_before = tracer_after.copy()
python_process.run(increment=900.0e-6)
override_after = base.fields.experiment_tracer.get(rank=0)
assert np.allclose(
    override_after,
    override_before + 900.0e-6,
    rtol=0.0,
    atol=np.spacing(900.0e-6),
)
assert python_process.parameters['increment'] == 1800.0e-6

# This dict-like assignment persistently changes later calls and steps.
python_process.parameters['increment'] = 900.0e-6
assert python_process.parameters['increment'] == 900.0e-6

# A disabled process is skipped by a complete step; enabling restores it.
python_process.disable()
disabled_before = base.fields.experiment_tracer.get(rank=0)
base.advance(steps=1)
disabled_after = base.fields.experiment_tracer.get(rank=0)
assert np.array_equal(disabled_before, disabled_after)
python_process.enable()
base.advance(steps=1)
enabled_after = base.fields.experiment_tracer.get(rank=0)
assert np.all(enabled_after > disabled_after)

# Move the Python process node immediately before another real scheme.
python_process.move(before='potential_temp_to_temp')
moved_order = tuple(
    row['name']
    for row in base.physics.describe('physics_before_coupler')
)
moved_index = moved_order.index('notebook_tracer_source')
assert moved_order[moved_index + 1] == 'potential_temp_to_temp'

# Move it back to immediately after Kessler. This changes only SuitePlan
# ordering; the cloudpickle payload and StatePool arrays do not move.
python_process.move(after='kessler')
restored_order = tuple(
    row['name']
    for row in base.physics.describe('physics_before_coupler')
)
restored_index = restored_order.index('notebook_tracer_source')
assert restored_order[restored_index - 1] == 'kessler'

# remove() unregisters the callback and deletes its SuitePlan node.
removed_python_process = python_process.remove()
names_after_remove = tuple(
    row['name']
    for row in base.physics.describe('physics_before_coupler')
)
assert 'notebook_tracer_source' not in names_after_remove

# Reinstall it for the retained-state and checkpoint examples.
python_process = base.physics.install_python(
    notebook_tracer_source,
    name='notebook_tracer_source',
    group='physics_before_coupler',
    after='kessler',
    writes=('experiment_tracer',),
    parameters={'increment': 900.0e-6},
)

# The disk checkpoint metadata retains python_process_inventory.
python_process_checkpoint = base.save()
installed_python_processes = base.status.details['python_processes']
{
    'process': python_process.name,
    'payload_hash': python_process.payload_hash,
    'writes': python_process.writes,
    'parameters': dict(python_process.parameters),
    'moved_immediately_before': moved_order[moved_index + 1],
    'moved_back_immediately_after': restored_order[restored_index - 1],
    'removed_once': removed_python_process['name'],
    'step': base.status.step,
    'checkpoint': python_process_checkpoint.path,
    'inventory': installed_python_processes,
}

{'process': 'notebook_tracer_source',
 'payload_hash': '6d6fadd662240434daeb18c8f349e95672470ebf9caeaf12d225e4022be28160',
 'writes': ('experiment_tracer',),
 'step': 4,
 'checkpoint': PosixPath('/glade/derecho/scratch/ruitong/pycam-sima/persistent_pool_trials/20260801-021434/models/cam-pool/base/checkpoints/checkpoint-1785572162841956125'),
 'inventory': ({'spec': {'schema_version': 1,
    'name': 'notebook_tracer_source',
    'payload_base64': 'gAWV6gIAAAAAAACMF2Nsb3VkcGlja2xlLmNsb3VkcGlja2xllIwOX21ha2VfZnVuY3Rpb26Uk5QoaACMDV9idWlsdGluX3R5cGWUk5SMCENvZGVUeXBllIWUUpQoSwJLAEsASwNLBUsDQ0aXAHwAZAEZAAAAAAAAAAAAfQJ8AmQCeAJ4AhkAAAAAAAAAAAB8AWoAAAAAAAAAAABkA3oFAAB6DQAAYwNjAjwAAABkAFMAlChOjBFleHBlcmltZW50X3RyYWNlcpSMCGJ1aWx0aW5zlIwIRWxsaXBzaXOUk5RHPrDG96C17Y10lIwQdGltZXN0ZXBfc2Vjb25kc5SFlIwGZmllbGRzlIwHY29udGV4dJSMBnRyYWNlcpSHlIxAL2dsYWRlL2RlcmVjaG8vc2NyYXRjaC9ydWl0b25nL3RtcC9pcHlrZXJuZWxfNTYzNTkvMTU3ODA1Mjk1Mi5weZSMFm5vdGVib29rX3RyYWNlcl9zb3VyY2WUjBZub3RlYm9va190cmFjZXJfc291cmNllEsBQy+AANgNE

## 6. Retain one reusable state and close the live model

`retain()` serializes each rank's local state into memory owned by that same MPI rank; only a small descriptor returns to Jupyter. After the snapshot is complete, `base.close()` releases the live model arrays while the retained bytes remain available for sequential restore.

In [ ]:
# One copy per MPI rank remains in that rank's memory after the live model closes.
retained_state = base.retain('after-python-process')
assert runtime.status['mpi_launch_count'] == 1
{
    'retained_snapshot': retained_state,
    'retained_bytes': retained_state.nbytes,
    'mpi_launch_count': runtime.status['mpi_launch_count'],
}

In [ ]:
# The live model registry contains the callback before retention.
base_inventory = base.status.details['python_processes']
assert any(
    item['spec']['name'] == 'notebook_tracer_source'
    for item in base_inventory
)
base_tracer_before = base.fields.experiment_tracer.get(rank=0)
base.physics.scheme(
    'notebook_tracer_source', group='physics_before_coupler'
).run()
base_tracer_after = base.fields.experiment_tracer.get(rank=0)
assert np.all(base_tracer_after > base_tracer_before)

# close() frees only the live arrays; the retained rank-local bytes remain.
base.close()
assert runtime.slots[0]['state'] == 'idle'
assert runtime.slots[0]['retained_snapshot_bytes'] == retained_state.nbytes
{
    'retained_payload_hash': base_inventory[0]['spec']['payload_hash'],
    'callback_ran_before_retention': True,
    'live_slot_state': runtime.slots[0]['state'],
    'retained_bytes_still_present': runtime.slots[0]['retained_snapshot_bytes'],
}

## 7. Optional durable disk-checkpoint comparison

The live base has now released the runtime's only model slot. This cell restores the durable checkpoint created earlier, verifies that its Python-process inventory survived, and closes the restored model before the rank-local retained-state experiments below. Use this path for cross-runtime restart or failure recovery.

In [10]:
with runtime.restore(
    'checkpoint-restart', python_process_checkpoint.path
) as restarted:
    restored_inventory = restarted.status.details['python_processes']
    assert any(
        item['spec']['name'] == 'notebook_tracer_source'
        for item in restored_inventory
    )
    restored_before = restarted.fields.experiment_tracer.get(rank=0)
    restarted.physics.scheme(
        'notebook_tracer_source', group='physics_before_coupler'
    ).run()
    restored_after = restarted.fields.experiment_tracer.get(rank=0)
    assert np.all(restored_after > restored_before)
    restored_step = restarted.status.step

assert runtime.status['mpi_launch_count'] == 1
{
    'restored_step': restored_step,
    'callback_restored': True,
    'mpi_launch_count': runtime.status['mpi_launch_count'],
}

{'restored_step': 4, 'callback_restored': True, 'mpi_launch_count': 1}

## 8. Restore one branch and run one selected scheme

The `control` model is reconstructed from rank-local memory in the same slot. This calls only Kessler: it does not execute the surrounding suite order, advance the model clock, or write a complete-step history record.

In [7]:
with runtime.restore_retained('control', retained_state) as control:
    control_temperature = control.fields.air_temperature.get(rank=0)
    control_step_before_kessler = control.status.step
    kessler_result = control.physics.scheme(
        'kessler',
        group='physics_before_coupler',
    ).run()
    control_status_after_kessler = control.status
    assert control_status_after_kessler.step == control_step_before_kessler
    control_worker = control.worker
    control_slot = control.slot_id
{
    'target_model': 'control',
    'target_worker': control_worker,
    'target_slot': control_slot,
    'step_unchanged': control_status_after_kessler.step,
    'kessler_result': kessler_result,
}

{'target_model': 'control',
 'target_worker': 'inproc://128.117.211.176/56359/6',
 'target_slot': 0,
 'step_unchanged': 4,
 'kessler_result': {'step': 4,
  'native_nstep': 4,
  'native_calls': 1397,
  'mpi_communicator_handle': -2080374782,
  'phase_status': {'runtime': 'model',
   'state': 'RUNNING',
   'last_phase': 'physics_timestep_initial',
   'last_scheme': 'physics_before_coupler.kessler@7',
   'last_scheme_group': 'physics_before_coupler',
   'next_phase': None,
   'sequence_safe': False,
   'step': 4,
   'native_nstep': 4},
  'scheme_status': {'last_scheme': 'physics_before_coupler.kessler@7',
   'last_scheme_group': 'physics_before_coupler',
   'sequence_safe': False,
   'groups': ('physics_before_coupler', 'physics_after_coupler'),
   'plan': {'schema_version': 2,
    'name': 'kessler',
    'source': '/glade/work/ruitong/pycam-sima/external/CAM-SIMA/src/physics/ncar_ccpp/suites/suite_kessler.xml',
    'sequence_safe': False,
    'groups': {'physics_before_coupler': {'kind': 

## 9. Reuse the same retained state for sequential branches

Only one model is live at a time. Each restore rebuilds private arrays from the same immutable rank-local snapshot, runs its experiment, and releases the slot for the next branch. No `qsub`, `mpiexec`, disk checkpoint, or parent rerun occurs.

In [8]:
with runtime.restore_retained('warm', retained_state) as warm:
    warm.fields.air_temperature += 1.0
    warm_temperature = warm.fields.air_temperature.get(rank=0)
    assert np.array_equal(warm_temperature, np.add(control_temperature, 1.0))
    removed_from_warm = warm.physics.remove_python('notebook_tracer_source')
    warm.advance(steps=1)
    warm_status = warm.status

with runtime.restore_retained('no-kessler', retained_state) as no_kessler:
    # This independently restored branch still has the Python callback.
    assert any(
        item['spec']['name'] == 'notebook_tracer_source'
        for item in no_kessler.status.details['python_processes']
    )
    no_kessler.physics.kessler.disable()
    no_kessler.advance(steps=1)
    no_kessler_status = no_kessler.status

assert runtime.status['mpi_launch_count'] == 1
{
    'warm_step': warm_status.step,
    'no_kessler_step': no_kessler_status.step,
    'warm_removed_python_process': removed_from_warm['name'],
    'same_reused_slot': control_slot,
    'runtime_after_branches': runtime.slots[0]['state'],
    'mpi_launch_count': runtime.status['mpi_launch_count'],
}

{'warm_step': 5,
 'no_kessler_step': 5,
 'warm_removed_python_process': 'notebook_tracer_source',
 'same_reused_slot': 0,
 'slot_after_branches': 'idle',
 'mpi_launch_count': 1}

## 10. Optional legacy multi-job comparison

Set the flag to `True` only when the Notebook runs in `execution_mode='pbs'`. This intentionally submits one legacy base job plus three child jobs. Compare this cell's Jupyter duration with the persistent-runtime cells above. It is disabled by default to prevent accidental extra PBS submissions.

In [9]:
run_legacy_comparison = False

if not run_legacy_comparison:
    legacy_result = 'skipped; set run_legacy_comparison=True for the real four-job comparison'
elif execution_mode != 'pbs':
    raise RuntimeError('legacy multi-model timing requires execution_mode="pbs"')
else:
    legacy_client = Client(
        processes=False,
        n_workers=4,
        threads_per_worker=1,
        dashboard_address=None,
    )
    legacy_experiments = DaskExperimentClient(
        legacy_client,
        config=config_path,
        initial_run_dir=initial_run_dir,
        run_root=experiment_root / 'legacy-benchmark',
        python_executable=repo / '.venv/bin/python',
        execution_mode='pbs',
    )
    try:
        with legacy_experiments.open_persistent('legacy-base') as legacy_base:
            legacy_base.advance(steps=2)
            control_plan = legacy_experiments.plan('legacy-control')
            no_kessler_plan = legacy_experiments.plan('legacy-no-kessler', experimental=True)
            no_kessler_plan.physics.kessler.disable()
            warm_plan = legacy_experiments.plan('legacy-warm')
            warm_plan.fields.edit('air_temperature', 'add', 1.0)

            legacy_children = legacy_experiments.fork_models(
                legacy_base,
                (control_plan, no_kessler_plan, warm_plan),
                close_parent=False,
            )
            with legacy_children:
                legacy_children.advance(steps=1)
                legacy_statuses = legacy_children.statuses
                legacy_child_job_ids = {
                    name: status.pbs_job_id
                    for name, status in legacy_statuses.items()
                }
            legacy_base_job_id = legacy_base.status.pbs_job_id

        legacy_result = {
            'base_pbs_job_id': legacy_base_job_id,
            'child_pbs_job_ids': legacy_child_job_ids,
        }
    finally:
        legacy_client.close()

legacy_result

'skipped; set run_legacy_comparison=True for the real four-job comparison'

## 11. Cleanup

Run this cell when finished. Dropping the retained state releases its rank-local bytes. Closing the runtime then stops the single MPI world, and closing the Dask client releases its two control workers.

In [11]:
if not retained_state.closed:
    retained_state.close()
runtime.close()
client.close()
{
    'retained_state_dropped': True,
    'runtime_closed': True,
    'dask_client_closed': True,
}

{'retained_state_dropped': True,
 'pool_closed': True,
 'dask_client_closed': True}